MINICPM

In [1]:
!pip install ollama


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import json
import re
import time

import ollama

# ============================================================
# Local Ollama configuration
# ============================================================
MODEL_NAME = "minicpm-v4.6:latest"
EXDARK_IMAGE_DIR = "ExDark/ExDark/"
OUTPUT_DIR = "ExDark/ExDarkAnnotation-minicpm"
FAILED_LOG_PATH = "ExDark/failed_images.txt"

client = ollama.Client()
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Plain-text prompt
SYSTEM_PROMPT = """You are a Computer Vision expert. Look at this image and describe the main object or scene in ONE short English sentence.

RULES:
- Output exactly one sentence.
- No quotes, no JSON, no markdown, no labels like "Description:".
- No introduction, no explanation, just the sentence itself."""


def log_failure(image_path, error_msg):
    with open(FAILED_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(f"{image_path} | Error: {error_msg}\n")


# ============================================================
# Cleanup helpers
# ============================================================


def clean_to_sentence(raw_text):
    text = raw_text.strip()

    # Take only the first line (model sometimes adds extra commentary below)
    text = text.splitlines()[0].strip()

    # Keep only up to the first sentence-ending punctuation, inclusive
    match = re.search(r"^[^.!?]*[.!?]", text)
    if match:
        text = match.group(0)

    text = text.strip().strip(quote_chars).strip()

    if not text:
        raise ValueError("Model returned empty output after cleaning.")

    return text


def wrap_as_annotation(sentence):
    """Build the JSON object with the control tokens around the sentence."""
    return {"description": f"<|desc_start|> {sentence} <|desc_end|>"}


# ============================================================
# Ollama call
# ============================================================
def call_model_with_retry(image_path, max_retries=5, base_delay=2.0):
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat(
                model=MODEL_NAME,
                messages=[
                    {
                        "role": "user",
                        "content": SYSTEM_PROMPT,
                        "images": [image_path],
                    }
                ],
                options={"temperature": 0.2},
            )
            return response["message"]["content"]

        except ollama.ResponseError as e:
            last_err = e
            delay = base_delay * attempt
            print(f"Ollama error: {e.error}. Retrying in {delay:.1f}s ({attempt}/{max_retries})...")
            time.sleep(delay)

        except Exception as e:
            last_err = e
            delay = base_delay * attempt
            print(f"Could not reach Ollama (is `ollama serve` running?). "
                  f"Retrying in {delay:.1f}s ({attempt}/{max_retries})...")
            time.sleep(delay)

    raise Exception(f"Failed after {max_retries} attempts. Last error: {last_err}")


# ============================================================
# Main loop
# ============================================================
def process_images_recursively(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue

            image_path = os.path.join(root, file)
            output_filename = os.path.splitext(file)[0] + ".json"
            output_path = os.path.join(OUTPUT_DIR, output_filename)

            if os.path.exists(output_path):
                continue

            print(f"Processing: {file}...")

            try:
                raw_text = call_model_with_retry(image_path)
                sentence = clean_to_sentence(raw_text)
                json_data = wrap_as_annotation(sentence)

                with open(output_path, "w", encoding="utf-8") as f:
                    json.dump(json_data, f, indent=4, ensure_ascii=False)

            except Exception as e:
                print(f"Skipping {file}. Permanent failure logged.")
                log_failure(image_path, str(e))
                continue


if __name__ == "__main__":
    print("Starting local annotation process...")
    process_images_recursively(EXDARK_IMAGE_DIR)
    print("Finished loop sequence!")

Starting local annotation process (plain text -> JSON in Python)...
Processing: 2015_00431.jpg...
